# Flexible, versioned score rules

This notebook is the source of truth for the declarative predicate language, user calibrations, and registered scoring strategies.

In [ ]:
#| default_exp rules

In [ ]:
#| export
"""Versioned, explainable score rules with one ligbase table per run."""

from __future__ import annotations

from dataclasses import dataclass
from decimal import Decimal, ROUND_HALF_UP
import json
from pathlib import Path
import re
from typing import Any, Callable, ClassVar, Mapping


class RuleValidationError(ValueError):
    """Raised when a rules file is ambiguous or internally inconsistent."""


DINUCLEOTIDES = frozenset(a + b for a in "ACGT" for b in "ACGT")


@dataclass(frozen=True)
class ScoreExplanation:
    ruleset_id: str
    ligation_dinucleotide: str
    base_score: float
    modifiers: tuple[str, ...]
    recalculated_score: float

    def as_dict(self) -> dict[str, Any]:
        return {
            "ruleset_id": self.ruleset_id,
            "ligation_dinucleotide": self.ligation_dinucleotide,
            "base_score": self.base_score,
            "modifiers": list(self.modifiers),
            "recalculated_score": self.recalculated_score,
        }


@dataclass(frozen=True)
class RuleSet:
    """A compact representation of the active score rules."""

    ruleset_id: str
    dimer_scores: Mapping[str, float]
    flags: Mapping[str, Mapping[str, Any]]
    scoring: Mapping[str, Any] | None = None

    _strategies: ClassVar[dict[str, Callable[[float, tuple[str, ...], Mapping[str, Any]], float]]] = {}

    @classmethod
    def from_mapping(cls, payload: Mapping[str, Any], *, ruleset_id: str | None = None) -> "RuleSet":
        canonical = payload.get("ligbase_scores")
        if canonical is None:
            raise RuleValidationError("Missing canonical ligbase_scores mapping")
        duplicate_keys = {"_ligbase", "ligation_dinucleotide"} & set(payload)
        if duplicate_keys:
            raise RuleValidationError("Rulesets may contain exactly one ligbase table; remove " + ", ".join(sorted(duplicate_keys)))
        if not isinstance(canonical, Mapping) or not canonical:
            raise RuleValidationError("Canonical ligation-dinucleotide scores must be a non-empty mapping")

        dimer_scores = {str(k).upper(): float(v) for k, v in canonical.items()}
        if set(dimer_scores) != DINUCLEOTIDES:
            missing = sorted(DINUCLEOTIDES - set(dimer_scores))
            extra = sorted(set(dimer_scores) - DINUCLEOTIDES)
            raise RuleValidationError(f"ligbase_scores must define exactly 16 dinucleotides; missing={missing}, extra={extra}")
        if any(not 0.0 <= value <= 1.0 for value in dimer_scores.values()):
            raise RuleValidationError("Every ligbase score must be between 0 and 1")
        flags = payload.get("modifiers", {})
        if not isinstance(flags, Mapping):
            raise RuleValidationError("flags must be a mapping")
        scoring = payload.get("scoring", {"strategy": "divide_by_modifier_count"})
        if not isinstance(scoring, Mapping):
            raise RuleValidationError("scoring must be a mapping")
        return cls(ruleset_id=ruleset_id or str(payload.get("ruleset_id", "unnamed-ruleset")), dimer_scores=dimer_scores, flags=flags, scoring=scoring)

    @classmethod
    def from_packaged(cls, chemistry: str) -> "RuleSet":
        """Load the one authoritative ligbase table shipped for a chemistry."""
        filenames = {"prime_5k": "prime_5k.json", "xenium_v1": "xenium_v1.json"}
        try:
            filename = filenames[chemistry]
        except KeyError as exc:
            raise RuleValidationError(f"No packaged ruleset for {chemistry!r}") from exc
        path = Path(__file__).with_name("rulesets") / filename
        return cls.from_mapping(json.loads(path.read_text()))

    def with_calibration(self, calibration: Mapping[str, Any]) -> "RuleSet":
        """Return a new versioned ruleset using a user-provided calibration.

        A calibration must identify itself through ``calibration_id`` and
        provide exactly one of:

        - ``ligbase_scores``: all 16 explicit scores, or
        - ``tiers``: named score tiers, each containing a score and its
          dinucleotides.  Tiers must jointly cover every dinucleotide once.

        Modifiers are deliberately inherited from the chemistry ruleset: a
        ligbase calibration cannot quietly alter QC policy.
        """
        calibration_id = calibration.get("calibration_id")
        if not isinstance(calibration_id, str) or not calibration_id.strip():
            raise RuleValidationError("User calibration requires a non-empty calibration_id")
        has_explicit = "ligbase_scores" in calibration
        has_tiers = "tiers" in calibration
        if has_explicit == has_tiers:
            raise RuleValidationError("User calibration must provide exactly one of ligbase_scores or tiers")
        if has_explicit:
            scores = calibration["ligbase_scores"]
        else:
            scores = _scores_from_tiers(calibration["tiers"])
        validated = RuleSet.from_mapping({
            "ruleset_id": f"{self.ruleset_id}+{calibration_id}",
            "ligbase_scores": scores,
            "modifiers": self.flags,
            "scoring": self.scoring,
        })
        return validated

    @classmethod
    def register_scoring_strategy(cls, name: str, strategy: Callable[[float, tuple[str, ...], Mapping[str, Any]], float]) -> None:
        """Register a reviewed Python strategy; user input cannot execute code."""
        if not name or not name.replace("_", "").isalnum():
            raise RuleValidationError("Strategy names must be non-empty alphanumeric snake_case")
        cls._strategies[name] = strategy

    def explain(self, candidate: Mapping[str, Any]) -> ScoreExplanation:
        ligbase = str(candidate.get("lig_dinuc") or candidate.get("ligation_dinucleotide") or "").upper()
        if ligbase not in self.dimer_scores:
            raise RuleValidationError(f"No {self.ruleset_id} base score for ligation dinucleotide {ligbase!r}")
        modifiers = tuple(name for name, spec in self.flags.items() if _flagged(candidate, name, spec))
        base = self.dimer_scores[ligbase]
        strategy_name = (self.scoring or {}).get("strategy", "divide_by_modifier_count")
        try:
            strategy = self._strategies[strategy_name]
        except KeyError as exc:
            raise RuleValidationError(f"Unknown scoring strategy {strategy_name!r}") from exc
        score = _round4(strategy(base, modifiers, self.scoring or {}))
        return ScoreExplanation(self.ruleset_id, ligbase, base, modifiers, score)


def _flagged(candidate: Mapping[str, Any], name: str, spec: Mapping[str, Any]) -> bool:
    """Evaluate composable, declarative predicate specifications."""
    if "all" in spec:
        return all(_flagged(candidate, name, clause) for clause in spec["all"])
    if "any" in spec:
        return any(_flagged(candidate, name, clause) for clause in spec["any"])
    if "not" in spec:
        return not _flagged(candidate, name, spec["not"])
    kind = spec.get("type", "")
    field = spec.get("field", spec.get("col"))
    if kind == "range":
        value = candidate.get(field)
        lo, hi = spec.get("min", spec.get("lo")), spec.get("max", spec.get("hi"))
        return value is not None and not (float(lo) <= float(value) <= float(hi))
    if kind == "threshold":
        value = candidate.get(field)
        cutoff = spec.get("value", spec.get("lt"))
        # NaN documents a deliberately disabled, not-yet-derived rule.
        if cutoff is None or (isinstance(cutoff, float) and cutoff != cutoff):
            return False
        return value is not None and float(value) < float(cutoff)
    if kind in {"boolean", "fail"}:
        return bool(candidate.get(field))
    if kind == "comparison":
        value, expected, operator = candidate.get(field), spec.get("value"), spec.get("operator")
        if value is None:
            return False
        operations = {
            "eq": lambda: value == expected,
            "ne": lambda: value != expected,
            "lt": lambda: float(value) < float(expected),
            "lte": lambda: float(value) <= float(expected),
            "gt": lambda: float(value) > float(expected),
            "gte": lambda: float(value) >= float(expected),
            "in": lambda: value in expected,
            "not_in": lambda: value not in expected,
        }
        try:
            return operations[operator]()
        except KeyError as exc:
            raise RuleValidationError(f"Unsupported comparison operator {operator!r}") from exc
    if kind == "regex":
        value = candidate.get(field)
        return value is not None and bool(re.search(str(spec["pattern"]), str(value)))
    raise RuleValidationError(f"Unsupported rule flag {name!r} of type {kind!r}")


def _round4(value: float) -> float:
    return float(Decimal(str(value)).quantize(Decimal("0.0001"), rounding=ROUND_HALF_UP))


def _scores_from_tiers(tiers: Mapping[str, Any]) -> dict[str, float]:
    if not isinstance(tiers, Mapping) or not tiers:
        raise RuleValidationError("tiers must be a non-empty mapping")
    scores: dict[str, float] = {}
    for tier_name, tier in tiers.items():
        if not isinstance(tier, Mapping) or "score" not in tier or "dinucleotides" not in tier:
            raise RuleValidationError(f"Tier {tier_name!r} requires score and dinucleotides")
        for dinucleotide in tier["dinucleotides"]:
            dimer = str(dinucleotide).upper()
            if dimer in scores:
                raise RuleValidationError(f"Dinucleotide {dimer} occurs in more than one tier")
            scores[dimer] = float(tier["score"])
    return scores


def _divide_by_modifier_count(base: float, modifiers: tuple[str, ...], _: Mapping[str, Any]) -> float:
    return base / (1 + len(modifiers))


def _multiplicative_penalty(base: float, modifiers: tuple[str, ...], scoring: Mapping[str, Any]) -> float:
    return base * float(scoring.get("factor_per_modifier", 0.5)) ** len(modifiers)


def _additive_penalty(base: float, modifiers: tuple[str, ...], scoring: Mapping[str, Any]) -> float:
    return max(0.0, min(1.0, base - float(scoring.get("penalty_per_modifier", 0.1)) * len(modifiers)))


RuleSet.register_scoring_strategy("divide_by_modifier_count", _divide_by_modifier_count)
RuleSet.register_scoring_strategy("multiplicative_penalty", _multiplicative_penalty)
RuleSet.register_scoring_strategy("additive_penalty", _additive_penalty)
